# Gravity Student Lab (Minimal View)

No coding steps are required in this notebook.

1. Choose model settings in **Gravity Spec Controls**.
2. Run **Run Gravity Regression**.
3. Read the coefficient table and diagnostics.

Use `gravity_student_lab_colab.ipynb` if you want to see and edit all code cells.


In [ ]:
# @title Gravity Spec Controls
preset_spec = "Canonical FE (OLS)" # @param ["Naive GDP gravity (no FE)", "Cost-proxy gravity (no FE)", "Canonical FE (OLS)", "Canonical FE (PPML)", "Custom"]
estimator = "OLS" # @param ["OLS", "PPML"]
include_fe = True # @param {type:"boolean"}
exporters = "ALL" # @param {type:"string"}
importers = "ALL" # @param {type:"string"}

try:
    import ipywidgets as widgets
except Exception as exc:
    raise ImportError(
        "ipywidgets is required for compact covariate controls. "
        "Install with `pip install ipywidgets` if needed."
    ) from exc

try:
    from IPython.display import display, Markdown
except Exception:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

BILATERAL_COVARIATES = [
    "ln_dist",
    "contig",
    "comlang_off",
    "rta",
    "dist",
    "comcol",
    "comleg_posttrans",
]
EXPORTER_COVARIATES = [
    "ln_gdp_o",
    "gdp_o",
    "ln_pop_o",
    "pop_o",
    "ln_gdpcap_o",
    "gdpcap_o",
    "wto_o",
    "eu_o",
]
IMPORTER_COVARIATES = [
    "ln_gdp_d",
    "gdp_d",
    "ln_pop_d",
    "pop_d",
    "ln_gdpcap_d",
    "gdpcap_d",
    "wto_d",
    "eu_d",
]
ALL_COVARIATES = BILATERAL_COVARIATES + EXPORTER_COVARIATES + IMPORTER_COVARIATES

PRESETS = {
    "Naive GDP gravity (no FE)": {
        "estimator": "OLS",
        "include_fe": False,
        "covariates": ["ln_dist", "ln_gdp_o", "ln_gdp_d"],
    },
    "Cost-proxy gravity (no FE)": {
        "estimator": "OLS",
        "include_fe": False,
        "covariates": ["ln_dist", "ln_gdp_o", "ln_gdp_d", "contig", "comlang_off", "rta"],
    },
    "Canonical FE (OLS)": {
        "estimator": "OLS",
        "include_fe": True,
        "covariates": ["ln_dist", "contig", "comlang_off", "rta"],
    },
    "Canonical FE (PPML)": {
        "estimator": "PPML",
        "include_fe": True,
        "covariates": ["ln_dist", "contig", "comlang_off", "rta"],
    },
}


def _build_group(covariates: list[str]) -> tuple[dict[str, widgets.Checkbox], widgets.VBox]:
    box_map = {}
    children = []
    for covariate in covariates:
        checkbox = widgets.Checkbox(
            value=False,
            description=covariate,
            indent=False,
            layout=widgets.Layout(width="98%"),
        )
        box_map[covariate] = checkbox
        children.append(checkbox)
    return box_map, widgets.VBox(children)


bilateral_map, bilateral_box = _build_group(BILATERAL_COVARIATES)
exporter_map, exporter_box = _build_group(EXPORTER_COVARIATES)
importer_map, importer_box = _build_group(IMPORTER_COVARIATES)

COVARIATE_WIDGETS = {}
COVARIATE_WIDGETS.update(bilateral_map)
COVARIATE_WIDGETS.update(exporter_map)
COVARIATE_WIDGETS.update(importer_map)

covariate_accordion = widgets.Accordion(children=[bilateral_box, exporter_box, importer_box])
covariate_accordion.set_title(0, "Bilateral covariates")
covariate_accordion.set_title(1, "Exporter-only covariates")
covariate_accordion.set_title(2, "Importer-only covariates")
covariate_accordion.selected_index = None

def _apply_covariate_selection(covariates: list[str]) -> None:
    selected = set(covariates)
    for name, checkbox in COVARIATE_WIDGETS.items():
        checkbox.value = name in selected

if preset_spec != "Custom":
    spec = PRESETS[preset_spec]
    estimator = spec["estimator"]
    include_fe = spec["include_fe"]
    _apply_covariate_selection(spec["covariates"])
    control_mode = "Preset mode: estimator, FE, and covariates are auto-applied from preset_spec."
else:
    control_mode = "Custom mode: estimator, FE, and covariates follow your selected checkboxes."


def get_selected_covariates() -> list[str]:
    if preset_spec != "Custom":
        return list(PRESETS[preset_spec]["covariates"])
    return [name for name in ALL_COVARIATES if COVARIATE_WIDGETS[name].value]


selected_covariates = get_selected_covariates()
if not selected_covariates:
    print("- warning: no covariate selected yet. In Custom mode, tick at least one checkbox before running the regression.")

print("Applied configuration")
print("- preset_spec:", preset_spec)
print("- estimator:", estimator)
print("- include_fe:", include_fe)
print("- selected_covariates:", ", ".join(selected_covariates))
print("- exporters filter:", exporters)
print("- importers filter:", importers)
print("- mode:", control_mode)
print("- note: comcur is not available in Gravity_V202010.dta and is not offered here.")

display(Markdown("**Covariate selection (click to expand each category):**"))
display(covariate_accordion)


In [ ]:
# @title Run Gravity Regression

# If Colab misses any package, uncomment the next lines.
# !pip -q install statsmodels
# !pip -q install ipywidgets

import io
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

try:
    from IPython.display import display, Markdown
except Exception:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

DATA_URL = "https://raw.githubusercontent.com/DTEcon/Teaching_International_Trade_PUBLIC/main/data/gravity_2019_student.csv"


def load_student_data() -> pd.DataFrame:
    try:
        data = pd.read_csv(DATA_URL)
        print(f"Loaded data from URL: {DATA_URL}")
        return data
    except Exception as exc:
        print("Could not load dataset from URL.")
        print(f"Reason: {exc}")
        print("Please upload gravity_2019_student.csv from LMS/repo.")
        try:
            from google.colab import files  # type: ignore
            uploaded = files.upload()
            if not uploaded:
                raise RuntimeError("No file uploaded.")
            first_name = next(iter(uploaded))
            data = pd.read_csv(io.BytesIO(uploaded[first_name]))
            print(f"Loaded uploaded file: {first_name}")
            return data
        except Exception as upload_exc:
            raise RuntimeError(
                "Data load failed from both URL and upload path."
            ) from upload_exc


df_full = load_student_data()
print(f"Rows: {len(df_full):,} | Columns: {len(df_full.columns)}")

EXPORTER_ONLY = {
    "gdp_o",
    "ln_gdp_o",
    "pop_o",
    "ln_pop_o",
    "gdpcap_o",
    "ln_gdpcap_o",
    "wto_o",
    "eu_o",
}
IMPORTER_ONLY = {
    "gdp_d",
    "ln_gdp_d",
    "pop_d",
    "ln_pop_d",
    "gdpcap_d",
    "ln_gdpcap_d",
    "wto_d",
    "eu_d",
}


def parse_iso_list(value: str) -> list[str]:
    value = str(value).strip()
    if value.upper() == "ALL" or value == "":
        return []
    return [item.strip().upper() for item in value.split(",") if item.strip()]


def stars(pvalue: float) -> str:
    if pd.isna(pvalue):
        return ""
    if pvalue < 0.01:
        return "***"
    if pvalue < 0.05:
        return "**"
    if pvalue < 0.1:
        return "*"
    return ""


def build_formula(dep_var: str, covariate_list: list[str], include_fe: bool) -> str:
    rhs_terms = list(covariate_list)
    if include_fe:
        rhs_terms.append("C(iso3_o)")
        rhs_terms.append("C(iso3_d)")
    if not rhs_terms:
        rhs_terms = ["1"]
    return dep_var + " ~ " + " + ".join(rhs_terms)


def fit_gravity_model(
    data: pd.DataFrame,
    estimator_name: str,
    covariate_list: list[str],
    include_fe: bool,
    exporter_filter: list[str],
    importer_filter: list[str],
):
    work = data.copy()

    if exporter_filter:
        work = work[work["iso3_o"].isin(exporter_filter)].copy()
    if importer_filter:
        work = work[work["iso3_d"].isin(importer_filter)].copy()

    required = ["iso3_o", "iso3_d", "pair_cluster", "trade_value"] + covariate_list
    missing = [col for col in required if col not in work.columns]
    if missing:
        raise ValueError(f"Missing columns in data: {missing}")

    if estimator_name == "OLS":
        work = work[work["trade_value"] > 0].copy()
        work["ln_trade"] = np.log(work["trade_value"])
        dep_var = "ln_trade"
        fit_data = work.dropna(subset=[dep_var] + covariate_list + ["pair_cluster"])
    elif estimator_name == "PPML":
        dep_var = "trade_value"
        fit_data = work.dropna(subset=covariate_list + ["pair_cluster", "trade_value"])
    else:
        raise ValueError("Estimator must be OLS or PPML")

    if fit_data.empty:
        raise ValueError("No rows left after filters and NA handling.")

    formula = build_formula(dep_var, covariate_list, include_fe)

    if estimator_name == "OLS":
        result = smf.ols(formula, data=fit_data).fit(
            cov_type="cluster",
            cov_kwds={"groups": fit_data["pair_cluster"]},
        )
    else:
        result = smf.glm(
            formula,
            data=fit_data,
            family=sm.families.Poisson(),
        ).fit(
            maxiter=200,
            cov_type="cluster",
            cov_kwds={"groups": fit_data["pair_cluster"]},
        )

    return result, fit_data, formula


def build_result_table(result, covariate_list: list[str]) -> pd.DataFrame:
    rows = []
    for var in covariate_list:
        coef = result.params.get(var, np.nan)
        se = result.bse.get(var, np.nan)
        pval = result.pvalues.get(var, np.nan)
        rows.append(
            {
                "variable": var,
                "coef": coef,
                "std_err": se,
                "p_value": pval,
                "stars": stars(pval),
            }
        )
    out = pd.DataFrame(rows)
    out["coef_with_stars"] = out.apply(
        lambda row: "" if pd.isna(row["coef"]) else f"{row['coef']:.4f}{row['stars']}",
        axis=1,
    )
    out["std_err_fmt"] = out["std_err"].apply(lambda x: "" if pd.isna(x) else f"({x:.4f})")
    return out


def absorbed_diagnostics(
    covariate_list: list[str],
    include_fe: bool,
    result_table: pd.DataFrame,
    fit_data: pd.DataFrame,
) -> list[str]:
    notes = []
    if include_fe:
        for var in covariate_list:
            if var in EXPORTER_ONLY and var in fit_data.columns:
                within_max = fit_data.groupby("iso3_o")[var].nunique(dropna=True).max()
                if pd.isna(within_max) or within_max <= 1:
                    notes.append(f"{var}: no within-exporter variation in sample; not separately identified from exporter FE.")
                else:
                    notes.append(f"{var}: exporter-only covariate with FE; interpretation is fragile.")

            if var in IMPORTER_ONLY and var in fit_data.columns:
                within_max = fit_data.groupby("iso3_d")[var].nunique(dropna=True).max()
                if pd.isna(within_max) or within_max <= 1:
                    notes.append(f"{var}: no within-importer variation in sample; not separately identified from importer FE.")
                else:
                    notes.append(f"{var}: importer-only covariate with FE; interpretation is fragile.")

    dropped = result_table[result_table["coef"].isna()]
    for var in dropped["variable"].tolist():
        notes.append(f"{var}: coefficient not estimated (dropped/collinear or no variation in sample).")

    if not notes:
        notes.append("No obvious FE-absorption/collinearity flags for the selected covariates.")
    return notes

selected_covariates = get_selected_covariates()
if not selected_covariates:
    raise ValueError("Please select at least one covariate.")

selected_exporters = parse_iso_list(exporters)
selected_importers = parse_iso_list(importers)

result, fit_data, formula = fit_gravity_model(
    data=df_full,
    estimator_name=estimator,
    covariate_list=selected_covariates,
    include_fe=include_fe,
    exporter_filter=selected_exporters,
    importer_filter=selected_importers,
)

summary_table = build_result_table(result, selected_covariates)
notes = absorbed_diagnostics(selected_covariates, include_fe, summary_table, fit_data)

print("Run configuration")
print("- mode:", control_mode)
print("- selected_covariates:", ", ".join(selected_covariates))
print()
print("Model formula:")
print(formula)
print()
print("Sample diagnostics:")
print(f"- N (estimation sample): {len(fit_data):,}")
print(f"- Number of exporters in sample: {fit_data['iso3_o'].nunique():,}")
print(f"- Number of importers in sample: {fit_data['iso3_d'].nunique():,}")
if estimator == "PPML":
    zero_share = (fit_data["trade_value"] == 0).mean()
    print(f"- Zero-flow share used by PPML: {100 * zero_share:.2f}%")

print()
print("Coefficient table (clustered SE by pair):")
display(summary_table[["variable", "coef_with_stars", "std_err_fmt", "p_value"]])

print("FE / collinearity diagnostics:")
for note in notes:
    print("-", note)
